In [ ]:
from langchain_naver import ChatClovaX

chat = ChatClovaX()

messages = [
    (
        "system",
        (
            "CLOVA Studio는 HyperCLOVA X 모델을 활용하여 AI 서비스를 손쉽게"
            " 만들 수 있는 개발 도구입니다."
        ),
    ),
    ("human", "CLOVA Studio가 무엇인가요?"),
]
ai_msg = chat.invoke(messages)
ai_msg

AIMessage(content='CLOVA Studio는 네이버의 초대규모(Hyperscale) 언어모델인 하이퍼클로바(HyperCLOVA) 기술을 바탕으로 만들어진 대화형 AI 서비스 제작 플랫폼입니다.\n\n사용자는 CLOVA Studio를 통해 다음과 같은 작업을 할 수 있습니다:\n\n1. **대화 시나리오 작성**: 사용자가 원하는 목적에 맞게 대화 시나리오를 작성할 수 있습니다. 예를 들어, 고객 상담 챗봇이나 상품 추천 봇 등을 만들 때 유용합니다.\n2. **AI 모델 학습 및 배포**: 작성한 시나리오와 함께 필요한 데이터를 입력하면, CLOVA Studio에서 자동으로 AI 모델을 학습시키고 이를 웹 또는 모바일 앱 등에 쉽게 배포할 수 있도록 지원합니다.\n3. **자연어 이해(NLU)**: 사용자의 발화 의도를 파악하고 이에 적절한 답변을 제공하도록 설계되어 있어 자연스러운 대화가 가능합니다.\n4. **멀티턴 대화**: 이전 대화를 기억하며 이어지는 질문에도 자연스럽게 대응할 수 있는 멀티턴 대화 기능을 제공합니다.\n\n이를 통해 기업들은 별도의 개발자나 데이터 과학자의 도움 없이도 손쉽게 자신의 비즈니스에 최적화된 대화형 AI 서비스를 구축하고 운영할 수 있습니다.\n\n또한, CLOVA Studio는 사용자 프라이버시를 보호하면서도 높은 성능을 유지하기 위해 다양한 기술과 정책을 적용하고 있으며, 지속적인 업데이트를 통해 더욱 향상된 기능과 안정성을 제공할 예정입니다. ', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 271, 'prompt_tokens': 39, 'total_tokens': 310, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_provider': 'openai', 'model_name': 'HCX-005',

In [2]:
from langchain_community.retrievers import WikipediaRetriever

retriever = WikipediaRetriever(lang="ko")

docs = retriever.invoke("오버워치")

print(docs[0].page_content[:400])

오버워치(영어: Overwatch)는 블리자드 엔터테인먼트가 개발 및 배급한 팀 기반의 1인칭 슈팅 게임이다. 2016년 5월 24일에는 마이크로소프트 윈도우, 플레이스테이션 4, 엑스박스 원으로 출시되었고, 2019년 10월 15일에는 닌텐도 스위치 버전으로 출시하였다.
"영웅 슈터"로 묘사되는 오버워치는 플레이어를 6명으로 구성된 두 팀에 배정했으며, 각 플레이어는 "영웅"이라고 알려진 대규모 캐릭터 명단에서 고유한 능력을 가진 캐릭터를 선택했다. 팀은 제한된 시간 내에 지도별 목표를 완료하기 위해 노력했다. 블리자드는 출시 후 새로운 캐릭터, 지도, 게임 모드를 모두 무료로 추가했으며, 플레이어가 치장 아이템을 구매할 수 있는 옵션 전리품 상자만 추가 비용으로 지불했다.
오버워치는 블리자드의 네 번째 


In [25]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableMap
from langchain_community.retrievers import WikipediaRetriever
from langchain_core.output_parsers import JsonOutputParser

import json

from langchain_naver import ChatClovaX

retriever = WikipediaRetriever(lang="ko")


def format_docs(docs):
    return "\n\n".join([doc.page_content for doc in docs])


prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
            너는 퀴즈 제작 도우미야.
            아래 정보를 이용해 JSON 형식으로 퀴즈를 만들어야 해.

            각 퀴즈에는 **4개의 선택지**가 있어야 하며, 오직 하나의 선택지만이 정답이어야해.

            JSON의 스키마는 다음과 같아:
            {output_schema}

            오로지 주어진 문서의 내용만을 바탕으로 퀴즈를 만들어야해.
            문서: {context}
            문항 수: {num_questions}
            난이도: {difficulty_level}

            출력은 반드시 **순수 JSON만** 포함해야 해.
            설명이나 추가 텍스트는 절대 포함하지 마.
            """,
        ),
    ]
)


output_schema = {
    "name": "make_quiz",
    "description": (
        """
            function that takes a list of questions and answers and returns a quiz
        """
    ),
    "parameters": {
        "type": "object",
        "properties": {
            "questions": {
                "type": "array",
                "items": {
                    "type": "object",
                    "properties": {
                        "question": {"type": "string"},
                        "answers": {
                            "type": "array",
                            "items": {
                                "type": "object",
                                "properties": {
                                    "answer": {"type": "string"},
                                    "correct": {"type": "boolean"},
                                },
                                "required": ["answer", "correct"],
                            },
                        },
                    },
                    "required": ["question", "answers"],
                },
            }
        },
        "required": ["questions"],
    },
}


chat = ChatClovaX(verbose=True)

chain = (
    RunnableMap(
        {
            "context": (lambda x: x["topic"]) | retriever | format_docs,
            "num_questions": lambda x: x["amount"],
            "difficulty_level": lambda x: x["level"],
            "output_schema": lambda _: json.dumps(
                output_schema, ensure_ascii=False
            ),
        }
    )
    | prompt
    | chat
    | JsonOutputParser()
)

result = chain.invoke({"topic": "인공지능", "amount": 3, "level": "중급"})

result

{'questions': [{'question': '인공지능의 정의에 가장 부합하는 것은?',
   'answers': [{'answer': '사진 속 물체를 식별하는 컴퓨터 프로그램', 'correct': False},
    {'answer': '인간의 지능을 모방하여 구현한 컴퓨터 시스템', 'correct': True},
    {'answer': '특정 작업을 빠르고 효율적으로 처리하는 소프트웨어', 'correct': False},
    {'answer': '인터넷 검색 결과를 제공하는 알고리즘', 'correct': False}]},
  {'question': '다음 중 약인공지능(weak AI)에 해당하지 않는 것은?',
   'answers': [{'answer': '음성 명령을 인식하여 실행하는 스마트 스피커', 'correct': False},
    {'answer': '바둑에서 최적의 수를 찾는 인공지능', 'correct': True},
    {'answer': '얼굴 인식 기술을 활용한 보안 시스템', 'correct': False},
    {'answer': '특정 상황에서만 작동하는 진단 시스템', 'correct': False}]},
  {'question': '강인공지능(AGI)의 목표는 무엇인가?',
   'answers': [{'answer': '다양한 분야에서 인간과 경쟁하기 위함', 'correct': False},
    {'answer': '인간처럼 생각하고 문제를 해결할 수 있는 능력 구현', 'correct': True},
    {'answer': '단순히 반복 작업을 자동화하기 위함', 'correct': False},
    {'answer': '특정 문제를 빠르게 해결하기 위한 도구 제공', 'correct': False}]}]}

In [18]:
content = result.content

from langchain_core.output_parsers import JsonOutputParser

parser = JsonOutputParser()

quiz = parser.parse(content)

quiz

{'questions': [{'question': '다음 중 약인공지능(weak AI)의 주요 목적에 해당하지 않는 것은?',
   'answers': [{'answer': '사진에서 물체를 찾는 시스템을 구축', 'correct': True},
    {'answer': '인간의 감정을 이해하고 공감하는 시스템 구축', 'correct': False}]},
  {'question': '강인공지능(AGI)의 구현을 부정하는 철학자의 이름 하나를 제시하시오.',
   'answers': [{'answer': '존 설', 'correct': True},
    {'answer': '다니엘 C. 데넷', 'correct': False}]},
  {'question': '생성형 AI 시스템에서 멀티모달(multimodal) 시스템의 특징이 아닌 것은?',
   'answers': [{'answer': '하나의 입력만 받는다.', 'correct': True},
    {'answer': '둘 이상의 입력을 동시에 받을 수 있다.', 'correct': False}]}]}

In [26]:
result = chain.invoke({"topic": "오버워치", "amount": 3, "level": "초급"})

result

{'questions': [{'question': '오버워치가 처음 출시된 플랫폼은 무엇인가요?',
   'answers': [{'answer': '마이크로소프트 윈도우', 'correct': True},
    {'answer': '닌텐도 스위치', 'correct': False},
    {'answer': '플레이스테이션 3', 'correct': False},
    {'answer': '엑스박스 360', 'correct': False}]},
  {'question': '오버워치에서 플레이어는 몇 명의 영웅을 선택할 수 있나요?',
   'answers': [{'answer': '20여 명', 'correct': True},
    {'answer': '10명', 'correct': False},
    {'answer': '5명', 'correct': False},
    {'answer': '40명 이상', 'correct': False}]},
  {'question': '오버워치의 후속작인 오버워치 2는 언제 발표되었나요?',
   'answers': [{'answer': '2019년', 'correct': True},
    {'answer': '2020년', 'correct': False},
    {'answer': '2018년', 'correct': False},
    {'answer': '2021년', 'correct': False}]}]}